# 焊缝缺陷检测模型训练（云端 / 无 Mac / 不依赖 Roboflow）

**本笔记本开箱即训**：从 GitHub Release 直连下载公开焊缝数据集 → 自动把类别对齐成 App 可用的 `porosity` / `crack` → 用 Ultralytics 预训练 `yolov8n.pt` 迁移训练 → 导出 Core ML。

> **为什么不用 Roboflow**：国内网络无法访问 roboflow.com（被墙）。本路线改用 GitHub 直连数据集（`huangyebiaoke/steel-pipe-weld-defect-detection`，Release 里的 zip 国内可直连），训练与导出均不需要 Mac、不耗 GitHub Actions 额度；只有最后 Xcode 编译 + 打 IPA 等额度恢复后由 CI 完成。
>
> **先行训 5 类**：`porosity`(气孔) / `crack`(裂纹) / `undercut`(咬边) / `overlap`(焊瘤) / `unfused`(未融合) —— 气孔数据充足，咬边仅 35 张偏少需盯 recall。
> - **余高(excess_weld_metal) 不靠视觉模型**：App 里余高由 LiDAR 剖面 `WeldProfileAnalyzer` 几何计算（更早的诊断已确认 2D 亮度判余高不可靠）。
> - **咬边(undercut) 已纳入训练**：该数据集仅 35 张偏少，云端训练后重点看其 recall，过低则先用 App 拍照自标补充。
>
> **训练平台（任选，笔记本通用）**：
> - 百度 AI Studio（aistudio.baidu.com，免费 V100，国内可达，Jupyter，推荐）→ 上传本 `.ipynb` 新建 Notebook 选 GPU 环境。
> - Google Colab（若你能打开）→ 上传本 `.ipynb`，运行时改 GPU。
> - Kaggle Notebook（kaggle.com/code，每周 30h 免费 T4）。

In [ ]:
# ===== 1. 安装依赖 =====
# AI Studio 持久化安装目录 external-libraries 默认不在内核 sys.path 中，
# 必须先把它加进去，否则装了也 import 不到（模板 cell-4 已提示此事）。
import sys, subprocess

EXTLIB = '/home/aistudio/external-libraries'
if EXTLIB not in sys.path:
    sys.path.append(EXTLIB)

def pip_(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', *args])

try:
    import torch
    print('torch 已就绪', torch.__version__)
except ImportError:
    print('torch 缺失 -> 重装（约2GB，几分钟）...')
    pip_('uninstall', '-y', 'torch', 'torchvision', 'torchaudio')
    pip_('install', 'ultralytics', 'coremltools')
    import torch

print('torch', torch.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU 不可用！请检查是否选了 V100 GPU 环境'
from ultralytics import YOLO
print('ultralytics 导入成功')

In [ ]:
# ===== 2. 配置 =====
# 先行训 5 类: porosity/crack/undercut/overlap/unfused（气孔数据充足,咬边仅35张偏少）。
# 余高由 App 内 LiDAR 几何计算；视觉模型训 5 类: porosity/crack/undercut/overlap/unfused。
TARGET_CLASSES = ['porosity', 'crack', 'undercut', 'overlap', 'unfused']

# 源数据集类别名 -> 目标类别（None 表示暂丢弃该框）
REMAP = {
    'air-hole': 'porosity',   # 气孔
    'crack':    'crack',      # 裂纹
    'bite-edge': 'undercut',  # 咬边 (仅35张,样本少,重点盯 recall)
    'overlap':   'overlap',   # 焊瘤
    'unfused':   'unfused',   # 未融合
}

# 源数据集 data.yaml 找不到时的兜底类别顺序（来自其 README 的 Label 0..7）
SOURCE_FALLBACK = ['air-hole','bite-edge','broken-arc','crack','hollow-bead','overlap','slag-inclusion','unfused']
print('目标类别:', TARGET_CLASSES)

## 3. 下载数据集（来自 GitHub Release，国内可直连）

数据集：`huangyebiaoke/steel-pipe-weld-defect-detection`（YOLO + VOC 双格式，含气孔/裂纹/咬边等 8 类）。Release 里的 zip 走 GitHub 直连，不需要账号、不被墙。

> **若下载失败（极少见）**：用浏览器打开 https://github.com/huangyebiaoke/steel-pipe-weld-defect-detection/releases/tag/1.0 手动下载 `steel-tube-dataset-all.zip`，再上传到笔记本根目录并重命名为 `raw/steel-tube-dataset-all.zip`，然后从下一格的解压开始跑。

In [ ]:
import os, urllib.request, zipfile

URL = "https://github.com/huangyebiaoke/steel-pipe-weld-defect-detection/releases/download/1.0/steel-tube-dataset-all.zip"
os.makedirs('raw', exist_ok=True)
zip_path = 'raw/steel-tube-dataset-all.zip'

if not os.path.exists(zip_path):
    print('下载数据集(来自 GitHub Release)...')
    urllib.request.urlretrieve(URL, zip_path)
    print('下载完成:', os.path.getsize(zip_path), 'bytes')
else:
    print('已存在，跳过下载')

print('解压...')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('raw')
print('解压完成')

## 4. 自动整理数据集（发现图片+标注 → 类别重映射 → 8:2 划分）

兼容 YOLO(.txt) 与 VOC(.xml) 两种标注：经 REMAP 把 air-hole→porosity、crack→crack、bite-edge→undercut、overlap→overlap、unfused→unfused 保留为 5 类，其余类别丢弃；自动 80/20 切分训练/验证集，生成 App 训练用的 `dataset/data.yaml`。

In [ ]:
import os, glob, yaml, shutil, hashlib, xml.etree.ElementTree as ET

name2newid = {n: i for i, n in enumerate(TARGET_CLASSES)}

# 尝试读取数据集自带 data.yaml 确定源类别名
src_names = None
for root, _, files in os.walk('raw'):
    for f in files:
        if f.endswith(('.yaml', '.yml')):
            try:
                cfg = yaml.safe_load(open(os.path.join(root, f)))
                if isinstance(cfg, dict) and isinstance(cfg.get('names'), list) and len(cfg['names']) >= 3:
                    src_names = {i: n for i, n in enumerate(cfg['names'])}
                    print('发现 data.yaml，源类别:', cfg['names'])
                    break
            except Exception:
                pass
    if src_names:
        break
if src_names is None:
    src_names = {i: n for i, n in enumerate(SOURCE_FALLBACK)}
    print('未找到 data.yaml，使用 README 硬编码源类别:', SOURCE_FALLBACK)

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.PNG')

def parse_yolo(line):
    p = line.split()
    if len(p) < 5:
        return None
    try:
        cid = int(p[0])
    except ValueError:
        return None
    sname = src_names.get(cid)
    tgt = REMAP.get(sname) if sname else None
    if tgt is None:
        return None
    return f"{name2newid[tgt]} {' '.join(p[1:])}"

def parse_voc(xml_path):
    out = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        sz = root.find('size')
        w = float(sz.find('width').text); h = float(sz.find('height').text)
        for obj in root.findall('object'):
            sname = obj.find('name').text.strip()
            tgt = REMAP.get(sname)
            if tgt is None:
                continue
            bb = obj.find('bndbox')
            xmin = float(bb.find('xmin').text); ymin = float(bb.find('ymin').text)
            xmax = float(bb.find('xmax').text); ymax = float(bb.find('ymax').text)
            cx = ((xmin + xmax) / 2) / w
            cy = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            out.append(f"{name2newid[tgt]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    except Exception as e:
        return []
    return out

for sp in ('train', 'val'):
    os.makedirs(f'dataset/images/{sp}', exist_ok=True)
    os.makedirs(f'dataset/labels/{sp}', exist_ok=True)

kept = {c: 0 for c in TARGET_CLASSES}

# 全局收集图片/标注，按 base name 配对 —— 兼容「同目录」与「images/ 和 labels/ 分离」两种结构
img_map, yolo_map, voc_map = {}, {}, {}
for root, _, files in os.walk('raw'):
    for f in files:
        fl = f.lower()
        base = os.path.splitext(f)[0]
        if fl.endswith(IMG_EXTS):
            img_map[base] = os.path.join(root, f)
        elif fl.endswith('.txt'):
            yolo_map[base] = os.path.join(root, f)
        elif fl.endswith('.xml'):
            voc_map[base] = os.path.join(root, f)

total = 0
for base, ip in img_map.items():
    ann = yolo_map.get(base)
    boxes = []
    if ann:                       # 优先 YOLO(.txt)
        for line in open(ann):
            r = parse_yolo(line.strip())
            if r:
                boxes.append(r)
    else:
        ann = voc_map.get(base)
        if ann:                   # 退回 VOC(.xml)
            boxes = parse_voc(ann)
    if not boxes:
        continue
    f = os.path.basename(ip)
    split = 'val' if (int(hashlib.md5(base.encode()).hexdigest(), 16) % 5 == 0) else 'train'
    shutil.copy(ip, os.path.join('dataset/images', split, f))
    with open(os.path.join('dataset/labels', split, base + '.txt'), 'w') as wf:
        wf.write('\n'.join(boxes) + '\n')
    total += 1
    for b in boxes:
        kept[TARGET_CLASSES[int(b.split()[0])]] += 1

print('保留图-标对:', total)
print('各类框数:', kept)
with open('dataset/data.yaml', 'w') as f:
    yaml.safe_dump({
        'path': 'dataset',
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(TARGET_CLASSES),
        'names': TARGET_CLASSES,
    }, f)
print('已生成 dataset/data.yaml ->', TARGET_CLASSES)


## 5. 迁移训练（自动调用网络预训练权重 yolov8n.pt）

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # 自动从 Ultralytics 服务器下载 COCO 预训练权重
results = model.train(
    data='dataset/data.yaml',
    task='detect',
    epochs=80, imgsz=640, batch=16,
    name='weld_defect_5cls',
    patience=20,          # 20 轮无提升早停
    augment=True,         # 小数据集防过拟合
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, seed=42,
)
print('训练完成')

In [ ]:
# 验证 mAP（气孔类别目标 map50 >= 0.85 再上机；裂纹样本少，关注其 recall）
metrics = model.val()
print('mAP50   :', round(metrics.box.map50, 4))
print('mAP50-95:', round(metrics.box.map, 4))

## 6. 导出 Core ML 模型

导出 `WeldDefectModel.mlmodel`（含 NMS）。coremltools 转换在云端完成；Xcode 编译 + 打包等 GitHub 额度恢复。

In [ ]:
model = YOLO('runs/detect/weld_defect_5cls/weights/best.pt')
exported = model.export(format='coreml', nms=True, int8=False, imgsz=640)
import os
if os.path.exists(exported):
    shutil.move(exported, 'WeldDefectModel.mlmodel')
    print('已导出 WeldDefectModel.mlmodel')

## 7. 下载与下一步

1. 左侧文件面板右键 `WeldDefectModel.mlmodel` → 下载到电脑。
2. 放入 iOS 工程 `WeldFatigueChecker/` 并勾选 Target Membership。
3. 通知开发者：本模型为 **5 类**（porosity / crack / undercut / overlap / unfused），改写 `MLDefectDetector.runModel` 解析 YOLO 输出（坐标/置信度/类别），`labelMap` 设为上述 5 个；**余高(excess_weld_metal) 继续由 LiDAR 几何计算**、ROI 闸门/NMS 不变。
4. 等 GitHub Actions 额度恢复 → commit 触发 CI 打包 → Sideloadly 侧载验证。

**后续可继续补**：`broken-arc`(断弧)/`hollow-bead`(夹珠)/`slag-inclusion`(夹渣) 这 3 类当前未纳入（非 App 疲劳评估最优先），需要时在 `TARGET_CLASSES`/`REMAP` 增加对应映射并重训。
